In [ ]:
# add parent to path
import sys
from pathlib import Path

sys.path.append(Path.cwd().parent.parent.as_posix())

from demo_and_analysis.plots.utils.wandb_utils import get_wandb_stats

run_id = "i67xqak6"

summary, hist, config = get_wandb_stats(
    run_id,
    skip_cache=True,  # set to True to skip cache and fetch from W&B API
    wandb_run_cache_path=Path("/mnt/labstore/bespoke_olap/wandb_cache"),
)

start_turn = 1000

if start_turn > 0:
    hist = hist[hist["turn"] >= start_turn]

ModuleNotFoundError: No module named 'plots.utils.wandb_utils'

In [5]:
hist

,_runtime,_step,_timestamp,agent_name,apply_patch/added_loc_count,apply_patch/create_count,apply_patch/deleted_loc_count,apply_patch/string,apply_patch/update_count,cached_tokens,...,validation/scale_factor,validation/sf20_all_queries_avg_speedup,validation/sf20_all_queries_data,validation/sf20_all_queries_plot_table,validation/sf20_all_queries_total_speedup,validation/skip_validate,validation/total_duckdb_runtime_ms,validation/total_impl_runtime_ms,validation/total_speedup,validation/trace_mode
0,9.131886,0,1.774627e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20.0,6.382239,{'_latest_artifact_path': 'wandb-client-artifa...,{'artifact_path': 'wandb-client-artifact://a6r...,4.569771,False,49703.871575,10876.666667,4.569771,False
1,9.137379,1,1.774627e+09,Bespoke Assistant (Pinning),NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9.274856,2,1.774627e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9.279525,3,1.774627e+09,Bespoke Assistant (Pinning),NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9.414815,4,1.774627e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2236,2210.561101,2236,1.774629e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20.0,NaN,NaN,NaN,NaN,False,5144.978370,239.666667,21.467225,False
2237,2217.939408,2237,1.774629e+09,Bespoke Assistant (Optim w. Tracing Stats (1)),NaN,NaN,NaN,NaN,NaN,175172.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2238,2226.113212,2238,1.774629e+09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20.0,NaN,NaN,NaN,NaN,False,10980.769473,1566.000000,7.011986,False
2239,2226.173141,2239,1.774629e+09,Bespoke Assistant (Optim w. Tracing Stats (1)),NaN,NaN,NaN,NaN,NaN,175363.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
list(hist.columns)

['_runtime',
 '_step',
 '_timestamp',
 'agent_name',
 'apply_patch/added_loc_count',
 'apply_patch/create_count',
 'apply_patch/deleted_loc_count',
 'apply_patch/string',
 'apply_patch/update_count',
 'cached_tokens',
 'code/loc',
 'code/snapshot_hash',
 'compaction/candidate_items',
 'compaction/output_items',
 'compile/error',
 'compile/truncated',
 'context_window_usage',
 'cost_usd',
 'current_prompt',
 'current_prompt_descriptor',
 'input_tokens',
 'llm/output_text',
 'output_tokens',
 'prompt_idx',
 'reasoning_tokens',
 'shell/commands',
 'shell/num_commands',
 'shell/outputs',
 'supervisor',
 'supervisor/approved',
 'tool/applypatch_count',
 'tool/compaction_count',
 'tool/compile_count',
 'tool/handoff_count',
 'tool/llm_count',
 'tool/shell_count',
 'tool/validate_count',
 'total/cached_tokens',
 'total/cost_usd',
 'total/input_tokens',
 'total/output_tokens',
 'total/reasoning_tokens',
 'total/runtime',
 'turn',
 'type',
 'validation/all_queries',
 'validation/avg_speedup',
 

In [7]:
import numpy as np
import pandas as pd

# Work on a copy with stable row positions
df = hist.reset_index(drop=False).copy()
if "index" in df.columns and "_row" not in df.columns:
    df = df.rename(columns={"index": "_row"})

# Prefer the canonical column, then fallback to other likely names
token_col_candidates = [
    "input_tokens",
    "prompt_tokens",
    "usage/input_tokens",
    "llm/input_tokens",
]
token_col = next((c for c in token_col_candidates if c in df.columns), None)

if token_col is None:
    fallback = [c for c in df.columns if "input" in c.lower() and "token" in c.lower()]
    if not fallback:
        raise ValueError("Could not find an input-token column in hist")
    token_col = fallback[0]

print(f"Using token column: {token_col}")

token_values = pd.to_numeric(df[token_col], errors="coerce")

# LLM entries are expected to have numeric token values; tool rows are often NaN
llm_mask = token_values.notna()
llm_positions = np.flatnonzero(llm_mask.to_numpy())

if len(llm_positions) < 2:
    raise ValueError("Need at least 2 non-NaN token entries to compute diffs")

prev_pos = llm_positions[:-1]
curr_pos = llm_positions[1:]
deltas = token_values.iloc[curr_pos].to_numpy() - token_values.iloc[prev_pos].to_numpy()

pairs = (
    pd.DataFrame(
        {
            "start_pos": prev_pos.astype(int),
            "end_pos": curr_pos.astype(int),
            "delta": deltas.astype(float),
        }
    )
    .dropna(subset=["delta"])
    .sort_values("delta", ascending=False, kind="stable")
)
pairs = pairs.reset_index(drop=True)

if pairs.empty:
    raise ValueError("No valid token deltas found")

top_k = min(5, len(pairs))
top_pairs = pairs.head(top_k).copy()

print()
print(f"Top {top_k} increases in input tokens between consecutive non-NaN entries")
display(top_pairs)


def _first_non_null(row: pd.Series, cols: list[str]) -> str:
    for c in cols:
        if c in row.index and pd.notna(row[c]) and str(row[c]).strip() != "":
            return str(row[c])
    return "<none>"


desc_cols = ["current_prompt_descriptor", "type", "agent_name"]
request_cols = ["current_prompt", "request", "prompt", "user_input"]

preferred_cols = [
    "_rank",
    "_segment_role",
    "turn",
    "_step",
    "type",
    "agent_name",
    "current_prompt_descriptor",
    "current_prompt",
    token_col,
    "output_tokens",
    "cached_tokens",
    "reasoning_tokens",
    "shell/num_commands",
    "shell/commands",
    "compaction/output_items",
    "compaction/candidate_items",
    "apply_patch/added_loc_count",
    "apply_patch/deleted_loc_count",
]

for rank, pair in top_pairs.iterrows():
    start_pos = int(pair["start_pos"])
    end_pos = int(pair["end_pos"])
    delta = float(pair["delta"])

    segment = df.iloc[start_pos : end_pos + 1].copy()
    segment["_rank"] = rank + 1
    segment["_segment_role"] = "between"
    segment.iloc[0, segment.columns.get_loc("_segment_role")] = "previous_llm"
    segment.iloc[-1, segment.columns.get_loc("_segment_role")] = "next_llm"

    prev_row = df.iloc[start_pos]
    next_row = df.iloc[end_pos]

    print()
    print("=" * 100)
    print(f"Rank #{rank + 1} | delta = {delta:,.0f} | rows {start_pos} -> {end_pos}")

    print("Previous request (before spike):")
    print("descriptor:", _first_non_null(prev_row, desc_cols))
    print("request:", _first_non_null(prev_row, request_cols)[:1200])
    print(f"{token_col}: {token_values.iloc[start_pos]:,.0f}")

    print("Following LLM call (with spike):")
    print("descriptor:", _first_non_null(next_row, desc_cols))
    print("request:", _first_non_null(next_row, request_cols)[:1200])
    print(f"{token_col}: {token_values.iloc[end_pos]:,.0f}")

    show_cols = [c for c in preferred_cols if c in segment.columns]
    # Keep all rows between both LLM calls, including rows with NaN token values.
    display(segment[show_cols])

Using token column: input_tokens

Top 5 increases in input tokens between consecutive non-NaN entries


,start_pos,end_pos,delta
0,935,937,13276.0
1,523,525,12650.0
2,2069,2071,12050.0
3,882,884,11185.0
4,32,34,10632.0



Rank #1 | delta = 13,276 | rows 935 -> 937
Previous request (before spike):
descriptor: llm
request: <none>
input_tokens: 8,729
Following LLM call (with spike):
descriptor: llm
request: <none>
input_tokens: 22,005


,_rank,_segment_role,turn,_step,type,agent_name,current_prompt_descriptor,current_prompt,input_tokens,output_tokens,cached_tokens,reasoning_tokens,shell/num_commands,shell/commands,compaction/output_items,compaction/candidate_items,apply_patch/added_loc_count,apply_patch/deleted_loc_count
935,1,previous_llm,935,935,llm,Bespoke Assistant (Optim w. Sample Plan (6)),NaN,NaN,8729.0,693.0,5526.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
936,1,between,936,936,shell,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,[cat db_loader.cpp],NaN,NaN,NaN,NaN
937,1,next_llm,937,937,llm,Bespoke Assistant (Optim w. Sample Plan (6)),NaN,NaN,22005.0,110.0,8728.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN



Rank #2 | delta = 12,650 | rows 523 -> 525
Previous request (before spike):
descriptor: llm
request: <none>
input_tokens: 10,245
Following LLM call (with spike):
descriptor: llm
request: <none>
input_tokens: 22,895


...

,_rank,_segment_role,turn,_step,type,agent_name,current_prompt_descriptor,current_prompt,input_tokens,output_tokens,cached_tokens,reasoning_tokens,shell/num_commands,shell/commands,compaction/output_items,compaction/candidate_items,apply_patch/added_loc_count,apply_patch/deleted_loc_count
2069,3,previous_llm,2069,2069,llm,Bespoke Assistant (Optim w. Tracing Stats (1)),NaN,NaN,58854.0,12024.0,58440.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2070,3,between,2070,2070,apply_patch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.0,23.0
2071,3,next_llm,2071,2071,llm,Bespoke Assistant (Optim w. Tracing Stats (1)),NaN,NaN,70904.0,1235.0,58853.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN



Rank #4 | delta = 11,185 | rows 882 -> 884
Previous request (before spike):
descriptor: llm
request: <none>
input_tokens: 17,179
Following LLM call (with spike):
descriptor: llm
request: <none>
input_tokens: 28,364


,_rank,_segment_role,turn,_step,type,agent_name,current_prompt_descriptor,current_prompt,input_tokens,output_tokens,cached_tokens,reasoning_tokens,shell/num_commands,shell/commands,compaction/output_items,compaction/candidate_items,apply_patch/added_loc_count,apply_patch/deleted_loc_count
882,4,previous_llm,882,882,llm,Bespoke Assistant (Optim w. Sample Plan (5)),NaN,NaN,17179.0,7071.0,12138.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
883,4,between,883,883,apply_patch,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0
884,4,next_llm,884,884,llm,Bespoke Assistant (Optim w. Sample Plan (5)),NaN,NaN,28364.0,88.0,17178.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN



Rank #5 | delta = 10,632 | rows 32 -> 34
Previous request (before spike):
descriptor: llm
request: <none>
input_tokens: 1,571
Following LLM call (with spike):
descriptor: Add Timings for Queries 1, 2, 3
request: # Objective
Add comprehensive tracing instrumentation to measure query execution timing and row flow. The goal is to produce actionable data that identifies bottlenecks and validates optimizations.

# Tracing Infrastructure
The file `trace.hpp` is already present in the workspace. Include it in any file that needs instrumentation:
```cpp
#include "trace.hpp"
```

All tracing code is compiled away to nothing unless `-DTRACE` is set. `TRACE_RESET()` and `TRACE_FLUSH()` are already inserted into `query_impl.cpp` automatically — do not add them again.

## Available Macros
```cpp
PROFILE_SCOPE("section_name");   // RAII timer — measures until end of enclosing scope
TRACE_COUNT("counter_name", n);  // emit a COUNT line immediately
```

## Output Format
Stable, machine-parsable outpu

,_rank,_segment_role,turn,_step,type,agent_name,current_prompt_descriptor,current_prompt,input_tokens,output_tokens,cached_tokens,reasoning_tokens,shell/num_commands,shell/commands,compaction/output_items,compaction/candidate_items,apply_patch/added_loc_count,apply_patch/deleted_loc_count
32,5,previous_llm,32,32,llm,Supervision Agent,NaN,NaN,1571.0,5.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
33,5,between,33,33,validate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
34,5,next_llm,34,34,llm,"Bespoke Assistant (Add Timings for Queries 1, ...","Add Timings for Queries 1, 2, 3",# Objective\nAdd comprehensive tracing instrum...,12203.0,81.0,11086.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
